In [1]:
#1 — Imports
import requests
import random
import re
import time
import copy
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#2 — Ollama connection and model
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "phi3:mini"

def ask_phi(prompt, temperature=0.7):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": 30
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"].strip()

test_response = ask_phi("Reply with exactly the number 5.", temperature=0)
print(test_response)

#3 — Experiment parameters
N_BUYERS = 11
N_SELLERS = 11
ROUNDS = 5

MIN_VALUE = 0.75
MAX_VALUE = 3.25
STEP = 0.25

EQUILIBRIUM_PRICE = 2.00
EQUILIBRIUM_QUANTITY = 6

MAX_DECISIONS_PER_PERIOD = 60
STAGNATION_LIMIT = 15

TEMPERATURE = 0.7
SCHEDULER_SEED = 42
VALUATION_SEED = 42

FINAL_CALLS = True

#4 — Reservation values and costs
buyer_values = [MAX_VALUE - i * STEP for i in range(N_BUYERS)]
seller_costs = [MIN_VALUE + i * STEP for i in range(N_SELLERS)]

print("Buyer values:", buyer_values)
print("Seller costs:", seller_costs)

#5 — Randomly assign private values/costs to trader IDs
valuation_rng = random.Random(VALUATION_SEED)

buyer_assignments = buyer_values.copy()
seller_assignments = seller_costs.copy()

valuation_rng.shuffle(buyer_assignments)
valuation_rng.shuffle(seller_assignments)

agents = {}

for i in range(N_BUYERS):
    agent_id = f"B{i+1}"
    agents[agent_id] = {
        "id": agent_id,
        "role": "buyer",
        "reservation": buyer_assignments[i]
    }

for i in range(N_SELLERS):
    agent_id = f"S{i+1}"
    agents[agent_id] = {
        "id": agent_id,
        "role": "seller",
        "reservation": seller_assignments[i]
    }

print("\nTRADER ASSIGNMENTS")
print(pd.DataFrame(agents.values()).to_string(index=False))

#6 — Persistent private trader memory
private_memory = {}

for agent_id in agents:
    private_memory[agent_id] = {
        "periods": [],
        "current_actions": [],
        "current_trades": [],
        "current_profit": 0.0,
        "total_profit": 0.0
    }

#7 — Public cross-period history
public_history = []

#8 — Create a fresh trading-period state
def new_period_state():
    return {
        "active": set(agents.keys()),
        "current_bids": {},
        "current_asks": {},
        "transactions": [],
        "decision_log": [],
        "market_tape": [],
        "decision_count": 0,
        "stagnation": 0,
        "period": None
    }

#9 — Compact public market state
def public_market_state(state):
    bids = sorted(
        state["current_bids"].items(),
        key=lambda x: (-x[1], x[0])
    )

    asks = sorted(
        state["current_asks"].items(),
        key=lambda x: (x[1], x[0])
    )

    current_prices = [
        round(t["Price"], 4)
        for t in state["transactions"]
    ]

    return {
        "standing_bids": bids,
        "standing_asks": asks,
        "current_prices": current_prices,
        "recent_tape": state["market_tape"][-20:],
        "previous_periods": public_history[-4:]
    }

#10 — Compact private trader memory
def private_memory_state(agent_id):
    memory = private_memory[agent_id]

    period_summary = []

    for p in memory["periods"][-4:]:
        period_summary.append({
            "Period": p["Period"],
            "Actions": p["Actions"][-6:],
            "Trades": p["Trades"],
            "Profit": p["Profit"]
        })

    return {
        "previous_periods": period_summary,
        "current_actions": memory["current_actions"][-6:],
        "current_trades": memory["current_trades"],
        "total_profit": round(memory["total_profit"], 4)
    }

5
Buyer values: [3.25, 3.0, 2.75, 2.5, 2.25, 2.0, 1.75, 1.5, 1.25, 1.0, 0.75]
Seller costs: [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25]

TRADER ASSIGNMENTS
 id   role  reservation
 B1  buyer         1.50
 B2  buyer         2.50
 B3  buyer         2.75
 B4  buyer         1.25
 B5  buyer         2.00
 B6  buyer         1.75
 B7  buyer         1.00
 B8  buyer         2.25
 B9  buyer         3.25
B10  buyer         3.00
B11  buyer         0.75
 S1 seller         1.75
 S2 seller         1.50
 S3 seller         1.25
 S4 seller         3.25
 S5 seller         2.00
 S6 seller         1.00
 S7 seller         2.50
 S8 seller         2.75
 S9 seller         0.75
S10 seller         2.25
S11 seller         3.00


In [2]:
#11 — Prompt builder: MINIMAL
PROMPT_MODE = "minimal"

def build_prompt(agent_id, state):
    agent = agents[agent_id]
    public = public_market_state(state)
    private = private_memory_state(agent_id)

    if agent["role"] == "buyer":
        role_text = f"""
You are buyer {agent_id}.
Your private maximum willingness to pay is {agent["reservation"]:.2f}.
You may purchase at most one unit in this trading period.
If you buy at price P, your profit is {agent["reservation"]:.2f} - P.
You cannot buy above your maximum willingness to pay.
"""
        action_text = """
You may submit a BID at or below your maximum willingness to pay.
You may ACCEPT the lowest standing ask if it is acceptable to you.
You may PASS.
"""
        format_text = "BID X, ACCEPT, or PASS"

    else:
        role_text = f"""
You are seller {agent_id}.
Your private minimum willingness to accept is {agent["reservation"]:.2f}.
You may sell at most one unit in this trading period.
If you sell at price P, your profit is P - {agent["reservation"]:.2f}.
You cannot sell below your minimum willingness to accept.
"""
        action_text = """
You may submit an ASK at or above your minimum willingness to accept.
You may ACCEPT the highest standing bid if it is acceptable to you.
You may PASS.
"""
        format_text = "ASK X, ACCEPT, or PASS"

    prompt = f"""
{role_text}
Other traders' private information is unknown to you.
Use the public market information available to you.

{action_text}

PUBLIC MARKET INFORMATION:
Standing bids: {public["standing_bids"]}
Standing asks: {public["standing_asks"]}
Current transaction prices: {public["current_prices"]}
Recent public market tape: {public["recent_tape"]}
Previous-period public summaries: {public["previous_periods"]}

YOUR PRIVATE MEMORY:
{private}

Respond with exactly one action:
{format_text}
Do not provide an explanation.
"""
    return prompt.strip()

In [3]:
#12 — Parse LLM action
def parse_action(response):
    text = response.strip().upper()

    if "ACCEPT" in text:
        return {"action": "ACCEPT", "price": None}

    bid_match = re.search(
        r"BID\s+([0-9]+(?:\.[0-9]+)?)",
        text
    )
    if bid_match:
        return {
            "action": "BID",
            "price": float(bid_match.group(1))
        }

    ask_match = re.search(
        r"ASK\s+([0-9]+(?:\.[0-9]+)?)",
        text
    )
    if ask_match:
        return {
            "action": "ASK",
            "price": float(ask_match.group(1))
        }

    if "PASS" in text:
        return {"action": "PASS", "price": None}

    return {"action": "INVALID", "price": None}

#13 — Add public market event
def add_market_event(state, event):
    state["market_tape"].append(event)

#14 — Find best standing ask
def best_ask(state):
    if not state["current_asks"]:
        return None

    return min(
        state["current_asks"].items(),
        key=lambda x: (x[1], x[0])
    )

#15 — Find best standing bid
def best_bid(state):
    if not state["current_bids"]:
        return None

    return max(
        state["current_bids"].items(),
        key=lambda x: (x[1], x[0])
    )

#16 — Execute a trade
def execute_trade(
    state,
    buyer_id,
    seller_id,
    price,
    initiating_agent,
    trade_type
):
    buyer_value = agents[buyer_id]["reservation"]
    seller_cost = agents[seller_id]["reservation"]

    buyer_profit = buyer_value - price
    seller_profit = price - seller_cost

    trade = {
        "Period": state["period"],
        "Buyer": buyer_id,
        "Seller": seller_id,
        "Price": round(price, 4),
        "Buyer Value": buyer_value,
        "Seller Cost": seller_cost,
        "Buyer Profit": round(buyer_profit, 4),
        "Seller Profit": round(seller_profit, 4),
        "Initiating Agent": initiating_agent,
        "Trade Type": trade_type
    }

    state["transactions"].append(trade)

    state["active"].discard(buyer_id)
    state["active"].discard(seller_id)

    state["current_bids"].pop(buyer_id, None)
    state["current_asks"].pop(seller_id, None)

    private_memory[buyer_id]["current_trades"].append(trade)
    private_memory[seller_id]["current_trades"].append(trade)

    private_memory[buyer_id]["current_profit"] += buyer_profit
    private_memory[seller_id]["current_profit"] += seller_profit

    add_market_event(
        state,
        f"TRADE: {buyer_id}-{seller_id} @ {price:.2f}"
    )

    return trade

#17 — Process one trader decision
def process_agent(agent_id, state):
    if agent_id not in state["active"]:
        return False

    agent = agents[agent_id]
    prompt = build_prompt(agent_id, state)

    response = ask_phi(
        prompt,
        temperature=TEMPERATURE
    )

    parsed = parse_action(response)

    state["decision_count"] += 1

    log_entry = {
        "Period": state["period"],
        "Decision": state["decision_count"],
        "Agent": agent_id,
        "Role": agent["role"],
        "Response": response,
        "Action": parsed["action"],
        "Price": parsed["price"]
    }

    state["decision_log"].append(log_entry)
    private_memory[agent_id]["current_actions"].append(log_entry)

    # PASS
    if parsed["action"] == "PASS":
        add_market_event(
            state,
            f"{agent_id} passes"
        )
        return False

    # INVALID
    if parsed["action"] == "INVALID":
        add_market_event(
            state,
            f"{agent_id} gives invalid response"
        )
        return False

    # BUYER
    if agent["role"] == "buyer":

        if parsed["action"] == "BID":
            bid = parsed["price"]

            if bid > agent["reservation"]:
                add_market_event(
                    state,
                    f"{agent_id} invalid BID {bid:.2f}"
                )
                return False

            old_bid = state["current_bids"].get(agent_id)

            if old_bid == bid:
                add_market_event(
                    state,
                    f"{agent_id} repeats BID {bid:.2f}"
                )
                return False

            state["current_bids"][agent_id] = bid

            if old_bid is None:
                add_market_event(
                    state,
                    f"{agent_id} submits BID {bid:.2f}"
                )
            else:
                add_market_event(
                    state,
                    f"{agent_id} revises BID {old_bid:.2f} -> {bid:.2f}"
                )

            ask = best_ask(state)

            if ask is not None and bid >= ask[1]:
                seller_id, ask_price = ask

                execute_trade(
                    state,
                    agent_id,
                    seller_id,
                    ask_price,
                    agent_id,
                    "Bid Crossed Ask"
                )

            return True

        if parsed["action"] == "ACCEPT":
            ask = best_ask(state)

            if ask is not None and ask[1] <= agent["reservation"]:
                seller_id, ask_price = ask

                execute_trade(
                    state,
                    agent_id,
                    seller_id,
                    ask_price,
                    agent_id,
                    "Buyer Accepted Ask"
                )

                return True

            add_market_event(
                state,
                f"{agent_id} attempted ACCEPT with no acceptable ask"
            )
            return False

    # SELLER
    if agent["role"] == "seller":

        if parsed["action"] == "ASK":
            ask = parsed["price"]

            if ask < agent["reservation"]:
                add_market_event(
                    state,
                    f"{agent_id} invalid ASK {ask:.2f}"
                )
                return False

            old_ask = state["current_asks"].get(agent_id)

            if old_ask == ask:
                add_market_event(
                    state,
                    f"{agent_id} repeats ASK {ask:.2f}"
                )
                return False

            state["current_asks"][agent_id] = ask

            if old_ask is None:
                add_market_event(
                    state,
                    f"{agent_id} submits ASK {ask:.2f}"
                )
            else:
                add_market_event(
                    state,
                    f"{agent_id} revises ASK {old_ask:.2f} -> {ask:.2f}"
                )

            bid = best_bid(state)

            if bid is not None and bid[1] >= ask:
                buyer_id, bid_price = bid

                execute_trade(
                    state,
                    buyer_id,
                    agent_id,
                    bid_price,
                    agent_id,
                    "Ask Crossed Bid"
                )

            return True

        if parsed["action"] == "ACCEPT":
            bid = best_bid(state)

            if bid is not None and bid[1] >= agent["reservation"]:
                buyer_id, bid_price = bid

                execute_trade(
                    state,
                    buyer_id,
                    agent_id,
                    bid_price,
                    agent_id,
                    "Seller Accepted Bid"
                )

                return True

            add_market_event(
                state,
                f"{agent_id} attempted ACCEPT with no acceptable bid"
            )
            return False

    return False

#18 — Create a randomized round-robin decision order
def decision_order(active_agents, rng):
    order = list(active_agents)
    rng.shuffle(order)
    return order

#19 — Run one trading period
def run_period(period_number):
    state = new_period_state()
    state["period"] = period_number

    rng = random.Random(
        SCHEDULER_SEED + period_number
    )

    print(f"\n{'='*70}")
    print(f"PERIOD {period_number}")
    print(f"{'='*70}")

    while (
        len(state["active"]) >= 2
        and state["decision_count"] < MAX_DECISIONS_PER_PERIOD
        and state["stagnation"] < STAGNATION_LIMIT
    ):
        order = decision_order(state["active"], rng)

        for agent_id in order:
            if len(state["active"]) < 2:
                break

            if state["decision_count"] >= MAX_DECISIONS_PER_PERIOD:
                break

            changed = process_agent(
                agent_id,
                state
            )

            if changed:
                state["stagnation"] = 0
            else:
                state["stagnation"] += 1

            if state["stagnation"] >= STAGNATION_LIMIT:
                break

    if state["decision_count"] >= MAX_DECISIONS_PER_PERIOD:
        close_reason = "decision limit reached"
    elif state["stagnation"] >= STAGNATION_LIMIT:
        close_reason = "stagnation limit reached"
    else:
        close_reason = "fewer than two active traders"

    print(
        f"Main market closed after "
        f"{state['decision_count']} decisions "
        f"({close_reason})."
    )

    # Final calls
    if FINAL_CALLS and len(state["active"]) >= 2:
        print("\nFINAL CALLS")

        final_order = decision_order(
            state["active"],
            rng
        )

        for agent_id in final_order:
            if agent_id not in state["active"]:
                continue

            if len(state["active"]) < 2:
                break

            process_agent(
                agent_id,
                state
            )

    print("\nTRADES")

    for trade in state["transactions"]:
        print(
            f"{trade['Buyer']}-{trade['Seller']} "
            f"@ {trade['Price']:.2f} "
            f"({trade['Trade Type']})"
        )

    if not state["transactions"]:
        print("No trades.")

    return state

In [4]:
#20 — Summarize public period information
def period_public_summary(state):
    prices = [
        t["Price"]
        for t in state["transactions"]
    ]

    return {
        "Period": state["period"],
        "Trades": len(state["transactions"]),
        "Transaction Prices": [
            round(p, 4)
            for p in prices
        ],
        "Average Price": (
            round(np.mean(prices), 4)
            if prices else None
        ),
        "Price Range": (
            round(min(prices), 4),
            round(max(prices), 4)
        ) if prices else None,
        "Decisions": state["decision_count"]
    }

#21 — Save private period memory
def save_private_period_memory(state):
    for agent_id in agents:
        memory = private_memory[agent_id]

        memory["periods"].append({
            "Period": state["period"],
            "Actions": memory["current_actions"].copy(),
            "Trades": memory["current_trades"].copy(),
            "Profit": round(
                memory["current_profit"],
                4
            )
        })

        memory["total_profit"] += (
            memory["current_profit"]
        )

        memory["current_actions"] = []
        memory["current_trades"] = []
        memory["current_profit"] = 0.0

#22 — Test initial market state
test_state = new_period_state()

print("Active traders:", len(test_state["active"]))
print("Standing bids:", test_state["current_bids"])
print("Standing asks:", test_state["current_asks"])
print("Transactions:", test_state["transactions"])

#23 — Run the five-period experiment
all_states = []
all_decision_logs = []
all_transactions = []

experiment_start = time.time()

for period in range(1, ROUNDS + 1):
    state = run_period(period)

    all_states.append(state)

    all_decision_logs.extend(
        state["decision_log"]
    )

    all_transactions.extend(
        state["transactions"]
    )

    summary = period_public_summary(state)
    public_history.append(summary)

    save_private_period_memory(state)

experiment_time = time.time() - experiment_start

transactions_df = pd.DataFrame(
    all_transactions
)

decisions_df = pd.DataFrame(
    all_decision_logs
)

print(
    f"\nTotal runtime: "
    f"{experiment_time/60:.2f} minutes"
)

print(
    f"Total decisions: "
    f"{len(decisions_df)}"
)

print(
    f"Total trades: "
    f"{len(transactions_df)}"
)

#24 — Maximum theoretical surplus
def max_surplus():
    buyers = sorted(
        [
            a["reservation"]
            for a in agents.values()
            if a["role"] == "buyer"
        ],
        reverse=True
    )

    sellers = sorted(
        [
            a["reservation"]
            for a in agents.values()
            if a["role"] == "seller"
        ]
    )

    surplus = 0

    for buyer, seller in zip(
        buyers,
        sellers
    ):
        if buyer > seller:
            surplus += buyer - seller

    return surplus

MAX_SURPLUS = max_surplus()

print(
    "Maximum positive-surplus allocation:",
    MAX_SURPLUS
)

#25 — Smith alpha
def smith_alpha(
    prices,
    equilibrium_price=EQUILIBRIUM_PRICE
):
    if len(prices) == 0:
        return np.nan

    prices = np.array(
        prices,
        dtype=float
    )

    return (
        100
        * np.sqrt(
            np.mean(
                (prices - equilibrium_price) ** 2
            )
        )
        / equilibrium_price
    )

#26 — Period diagnostics
period_results = []

for state in all_states:
    prices = [
        t["Price"]
        for t in state["transactions"]
    ]

    actual_surplus = sum(
        t["Buyer Profit"]
        + t["Seller Profit"]
        for t in state["transactions"]
    )

    period_results.append({
        "Period": state["period"],
        "Trades": len(
            state["transactions"]
        ),
        "Average Price": (
            round(
                np.mean(prices),
                4
            )
            if prices else np.nan
        ),
        "Alpha": (
            round(
                smith_alpha(prices),
                4
            )
            if prices else np.nan
        ),
        "Surplus": round(
            actual_surplus,
            4
        ),
        "Efficiency (%)": round(
            100
            * actual_surplus
            / MAX_SURPLUS,
            4
        ) if MAX_SURPLUS > 0 else np.nan,
        "Decisions": state[
            "decision_count"
        ],
        "Stagnation": state[
            "stagnation"
        ]
    })

period_results_df = pd.DataFrame(
    period_results
)

print("\nPERIOD RESULTS")
print(
    period_results_df.to_string(
        index=False
    )
)

#27 — Transaction diagnostics
if not transactions_df.empty:
    transactions_df[
        "Total Surplus"
    ] = (
        transactions_df["Buyer Profit"]
        + transactions_df["Seller Profit"]
    )

    transactions_df[
        "Positive Surplus"
    ] = (
        transactions_df["Buyer Value"]
        > transactions_df["Seller Cost"]
    )

    print("\nTRANSACTION DIAGNOSTICS")

    print(
        transactions_df[
            [
                "Period",
                "Buyer",
                "Seller",
                "Price",
                "Buyer Value",
                "Seller Cost",
                "Buyer Profit",
                "Seller Profit",
                "Total Surplus",
                "Positive Surplus",
                "Trade Type"
            ]
        ].to_string(index=False)
    )

#28 — Overall summary
overall_summary = {
    "Total Trades": len(
        transactions_df
    ),
    "Total Decisions": len(
        decisions_df
    ),
    "Average Transaction Price": (
        transactions_df["Price"].mean()
        if not transactions_df.empty
        else np.nan
    ),
    "Overall Surplus": (
        transactions_df[
            "Total Surplus"
        ].sum()
        if not transactions_df.empty
        else 0
    ),
    "Overall Efficiency (%)": (
        100
        * transactions_df[
            "Total Surplus"
        ].sum()
        / (MAX_SURPLUS * ROUNDS)
        if not transactions_df.empty
        else 0
    ),
    "Runtime (minutes)": (
        experiment_time / 60
    )
}

print("\nOVERALL SUMMARY")

for key, value in overall_summary.items():
    if isinstance(value, (float, np.floating)):
        print(
            f"{key}: {value:.4f}"
        )
    else:
        print(
            f"{key}: {value}"
        )

#29 — Store experiment outputs
experiment_outputs = {
    "transactions": transactions_df,
    "decisions": decisions_df,
    "period_results": period_results_df,
    "states": all_states,
    "public_history": public_history,
    "private_memory": private_memory
}

print("\nExperiment complete.")

Active traders: 22
Standing bids: {}
Standing asks: {}
Transactions: []

PERIOD 1
Main market closed after 37 decisions (stagnation limit reached).

FINAL CALLS

TRADES
B8-S10 @ 2.25 (Seller Accepted Bid)
B2-S3 @ 2.50 (Ask Crossed Bid)
B9-S9 @ 2.75 (Seller Accepted Bid)
B3-S2 @ 2.75 (Seller Accepted Bid)
B10-S1 @ 2.75 (Seller Accepted Bid)
B5-S5 @ 2.00 (Seller Accepted Bid)

PERIOD 2
Main market closed after 50 decisions (stagnation limit reached).

FINAL CALLS

TRADES
B9-S2 @ 2.75 (Seller Accepted Bid)
B8-S1 @ 2.25 (Seller Accepted Bid)
B5-S3 @ 2.00 (Ask Crossed Bid)
B6-S9 @ 1.75 (Seller Accepted Bid)

PERIOD 3
Main market closed after 39 decisions (stagnation limit reached).

FINAL CALLS

TRADES
B5-S1 @ 2.00 (Seller Accepted Bid)
B1-S9 @ 1.50 (Seller Accepted Bid)
B2-S3 @ 2.50 (Seller Accepted Bid)

PERIOD 4
Main market closed after 41 decisions (stagnation limit reached).

FINAL CALLS

TRADES
B9-S2 @ 2.50 (Seller Accepted Bid)
B2-S3 @ 2.50 (Seller Accepted Bid)

PERIOD 5
Main market